# Data Understanding

In [1]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [2]:
print(df.shape)
print(df['sentiment'].value_counts())

(50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


# NLP Preprocessing

In [3]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    # lowercase
    text = text.lower()
    # removing html tags
    text = re.sub(r'<.*?>', '', text)
    # removing punctuations
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    # tokenization
    words = word_tokenize(text)
    # stopword removal
    words = [word for word in words if word not in stop_words]
    # lemmatization
    words = [lemmatizer.lemmatize(word) for word in words]
    # joining the words back
    return " ".join(words)

df['cleaned_review'] = df['review'].apply(preprocess)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Kashish\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Kashish\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Kashish\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Kashish\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
df[['review', 'cleaned_review']].head()

,review,cleaned_review
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode hoo...
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically family little boy jake think zombie ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visually stunnin...


# Feature Engineering

In [5]:
# Bag of Words(BoW)
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['cleaned_review'])

In [6]:
# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['cleaned_review'])

In [7]:
y = df['sentiment']
print(X_bow.shape)
print(X_tfidf.shape)

(50000, 5000)
(50000, 5000)


# Model Building

In [8]:
# train-test split-------------------------
from sklearn.model_selection import train_test_split

X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, y, test_size=0.2, random_state=42)
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

# logistic regression----------------------
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_bow, y_train)
y_pred_lr = lr.predict(X_test_bow)

# naive bayes------------------------------
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train_bow, y_train)
y_pred_nb = nb.predict(X_test_bow)

# decision tree-----------------------------
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train_bow, y_train)
y_pred_dt = dt.predict(X_test_bow)

# TF-IDF + logistic regression--------------
lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train)

y_pred_tfidf = lr_tfidf.predict(X_test_tfidf)
                                                     

# Model Evaluation

In [9]:
# Evaluation---------------------------------
from sklearn.metrics import classification_report

print("Logistic Regression:\n", classification_report(y_test, y_pred_lr))
print("Naive Bayes:\n", classification_report(y_test, y_pred_nb))
print("Decision Tree:\n", classification_report(y_test, y_pred_dt))
print("TF-IDF + Logistic Regression:\n", classification_report(y_test, y_pred_tfidf))

Logistic Regression:
               precision    recall  f1-score   support

    negative       0.88      0.87      0.87      4961
    positive       0.87      0.88      0.88      5039

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000

Naive Bayes:
               precision    recall  f1-score   support

    negative       0.85      0.85      0.85      4961
    positive       0.85      0.85      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000

Decision Tree:
               precision    recall  f1-score   support

    negative       0.71      0.72      0.72      4961
    positive       0.72      0.71      0.72      5039

    accuracy                           0.72     10000
   macro avg       0.72      0.72      0.72     10000
weighted avg       0.7

# Comparison & Insights

Final Results Summary (% accuracy):

BoW + Logistic Regression: 87%

Naive Bayes: 85%

Decision Tree: 72%

TF-IDF + Logistic Regression: 89%

----------------------------------------------

TF-IDF outperformed BoW because it assigns importance to meaningful words while reducing the weight of common words. Logistic Regression performed best among all models due to its effectiveness with high-dimensional sparse data.

Naive Bayes performed well but slightly lower due to its assumption of feature independence. Decision Tree showed the lowest performance because it struggles with sparse text data and tends to overfit.